# AQuA Full V10 Analysis (v2)

Initial analysis notebook for the `full_aqua_mc_mistral7b_instruct_V10` run. This notebook loads the run artifacts, explains the dataset and probe artifacts, then moves into question-level sycophancy analysis.

In [ ]:
import json
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_style("white")
plt.rcParams.update({
    "axes.titlesize": 18,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

PALETTE = {
    "neutral": "#4c4c4c",
    "suggest_correct": "#73b3ab",
    "doubt_correct": "#4f6d7a",
    "incorrect_suggestion": "#d4651a",
}
TEMPLATE_LABELS = {
    "neutral": "Neutral",
    "suggest_correct": "Suggest Correct",
    "doubt_correct": "Doubt Correct",
    "incorrect_suggestion": "Incorrect Suggestion",
}

RUN_DIR = Path("/Users/itaishapira/Desktop/R1_knowledge_gap/LLMsKnow/results/sycophancy_bias_probe/mistralai_Mistral_7B_Instruct_v0_2/aqua_mc/full_aqua_mc_mistral7b_instruct_V10")
RUN_NAME = RUN_DIR.name

run_dir = RUN_DIR
assert run_dir.exists(), f"Run directory not found: {run_dir}"

sampled_responses = pd.read_csv(run_dir / "sampled_responses.csv")
final_tuples = pd.read_csv(run_dir / "final_tuples.csv")
summary_by_question = pd.read_csv(run_dir / "summary_by_question.csv")
sampling_records = pd.read_json(run_dir / "sampling_records.jsonl", lines=True)

run_config = json.loads((run_dir / "run_config.json").read_text())
probe_metadata = json.loads((run_dir / "probe_metadata.json").read_text())
sampling_manifest = json.loads((run_dir / "sampling_manifest.json").read_text())
status = json.loads((run_dir / "status.json").read_text())
run_log = (run_dir / "run.log").read_text()
probe_model_files = sorted((run_dir / "probe_models").glob("*.pkl"))

artifact_overview = pd.DataFrame(
    [
        {"artifact": "sampled_responses.csv", "kind": "csv", "rows": len(sampled_responses)},
        {"artifact": "final_tuples.csv", "kind": "csv", "rows": len(final_tuples)},
        {"artifact": "summary_by_question.csv", "kind": "csv", "rows": len(summary_by_question)},
        {"artifact": "sampling_records.jsonl", "kind": "jsonl", "rows": len(sampling_records)},
        {"artifact": "run_config.json", "kind": "json", "rows": None},
        {"artifact": "probe_metadata.json", "kind": "json", "rows": None},
        {"artifact": "sampling_manifest.json", "kind": "json", "rows": None},
        {"artifact": "status.json", "kind": "json", "rows": None},
        {"artifact": "run.log", "kind": "text", "rows": None},
        {"artifact": "probe_models/", "kind": "directory", "rows": len(probe_model_files)},
    ]
)

display(artifact_overview)
print(f"Run directory: {run_dir}")
print(f"Probe model files: {len(probe_model_files)}")
print(f"Run status: {status.get('status', 'unknown')}")


## 1. AQuA dataset

### Dataset Shape And Labels

Definitions used below:

- **Question**: one unique `question_id`
- **Prompt**: one unique `(question_id, template_type)` pair
- **Responses per prompt**: number of repeated draws for the same prompt
- **Labeling statistics**: counts and rates for `correct`, `incorrect`, `ambiguous`, and `usable_for_metrics`


In [ ]:
n_questions = sampled_responses['question_id'].nunique()
n_prompts = sampled_responses[['question_id', 'template_type']].drop_duplicates().shape[0]
responses_per_prompt = sampled_responses.groupby(['question_id', 'template_type']).size()

usable_mask = sampled_responses['usable_for_metrics'].astype(str) == 'True'
correct_mask = sampled_responses['correctness'].astype(str).isin(['1', '1.0'])
incorrect_mask = sampled_responses['correctness'].astype(str).isin(['0', '0.0'])
ambiguous_mask = sampled_responses['grading_status'] == 'ambiguous'

label_summary = pd.DataFrame(
    [
        {'metric': 'questions', 'value': n_questions},
        {'metric': 'prompts', 'value': n_prompts},
        {'metric': 'responses', 'value': len(sampled_responses)},
        {'metric': 'responses_per_prompt_min', 'value': int(responses_per_prompt.min())},
        {'metric': 'responses_per_prompt_max', 'value': int(responses_per_prompt.max())},
        {'metric': 'responses_per_prompt_mean', 'value': round(float(responses_per_prompt.mean()), 2)},
        {'metric': 'correct', 'value': int(correct_mask.sum())},
        {'metric': 'incorrect', 'value': int(incorrect_mask.sum())},
        {'metric': 'ambiguous', 'value': int(ambiguous_mask.sum())},
        {'metric': 'usable_for_metrics', 'value': int(usable_mask.sum())},
    ]
)

rate_summary = pd.DataFrame(
    [
        {'rate': 'usable_rate', 'value': usable_mask.mean()},
        {'rate': 'correct_rate_overall', 'value': correct_mask.mean()},
        {'rate': 'incorrect_rate_overall', 'value': incorrect_mask.mean()},
        {'rate': 'ambiguous_rate_overall', 'value': ambiguous_mask.mean()},
        {'rate': 'accuracy_over_usable_only', 'value': correct_mask[usable_mask].mean()},
    ]
)
rate_summary['value'] = rate_summary['value'].round(4)

grading_breakdown = (
    sampled_responses['grading_status']
    .value_counts(dropna=False)
    .rename_axis('grading_status')
    .reset_index(name='count')
)

display(label_summary)
display(rate_summary)
display(grading_breakdown)


### Format Compliance

The strict MC contract for this run is that responses should be exactly one line in the form `Answer: <LETTER>`. The cells below summarize how often the model follows that format and what kinds of deviations remain.

In [ ]:
response_text = sampled_responses['response_raw'].fillna('').astype(str)
starts_with_answer_mask = sampled_responses['starts_with_answer_prefix'].astype(str) == 'True'
strict_exact_mask = sampled_responses['strict_format_exact'].astype(str) == 'True'
multiline_mask = response_text.str.contains(r'\n', regex=True)

def classify_format_response(text: str) -> str:
    text = str(text or '').strip()
    if re.fullmatch(r'Answer\s*:\s*[A-Za-z]', text):
        return 'exact_answer_letter_only'
    if not text.startswith('Answer:'):
        return 'no_answer_prefix'
    remainder = text[len('Answer:'):].strip()
    if '\n' in text:
        return 'answer_plus_extra_text'
    if re.fullmatch(r'\(?[A-Za-z]', remainder):
        return 'partial_or_truncated_answer'
    if re.fullmatch(r'[IVXLCM]+\.?', remainder):
        return 'noncanonical_roman_answer'
    if re.fullmatch(r'[0-9.]+', remainder):
        return 'noncanonical_numeric_answer'
    return 'other_nonexact_answer'

format_category = response_text.map(classify_format_response)

compliance_summary = pd.DataFrame(
    [
        {'metric': 'total_responses', 'value': len(sampled_responses)},
        {'metric': 'starts_with_answer_prefix', 'value': int(starts_with_answer_mask.sum())},
        {'metric': 'strict_format_exact', 'value': int(strict_exact_mask.sum())},
        {'metric': 'non_exact_but_starts_with_answer', 'value': int((starts_with_answer_mask & ~strict_exact_mask).sum())},
        {'metric': 'does_not_start_with_answer', 'value': int((~starts_with_answer_mask).sum())},
        {'metric': 'multiline_responses', 'value': int(multiline_mask.sum())},
    ]
)

compliance_rates = pd.DataFrame(
    [
        {'rate': 'starts_with_answer_prefix_rate', 'value': starts_with_answer_mask.mean()},
        {'rate': 'strict_format_exact_rate', 'value': strict_exact_mask.mean()},
        {'rate': 'non_exact_but_starts_with_answer_rate', 'value': (starts_with_answer_mask & ~strict_exact_mask).mean()},
        {'rate': 'no_answer_prefix_rate', 'value': (~starts_with_answer_mask).mean()},
        {'rate': 'multiline_response_rate', 'value': multiline_mask.mean()},
    ]
)
compliance_rates['value'] = compliance_rates['value'].round(4)

format_breakdown = (
    format_category.value_counts()
    .rename_axis('format_category')
    .reset_index(name='count')
)
format_breakdown['fraction'] = (format_breakdown['count'] / len(sampled_responses)).round(4)

noncompliant_examples = sampled_responses.loc[~strict_exact_mask, ['record_id', 'question_id', 'template_type', 'response_raw']].copy()
noncompliant_examples['format_category'] = format_category[~strict_exact_mask].values
noncompliant_examples = noncompliant_examples.groupby('format_category', as_index=False).first()

display(compliance_summary)
display(compliance_rates)
display(format_breakdown)
display(noncompliant_examples)


### Probe Artifacts

This run writes one probe model file per candidate layer, plus one final retrained probe per probe family.

- one **selection** model for every candidate layer
- one **best retrained** model for each probe family

There are `4` probe families in this run:

- `probe_no_bias` for neutral prompts
- `probe_bias_incorrect_suggestion`
- `probe_bias_doubt_correct`
- `probe_bias_suggest_correct`

The layer grid for this run is taken from `run_config['probe_layer_min']..run_config['probe_layer_max']`.

So the total number of saved probe files is:

- `4 × (# candidate layers)` selection models
- `4 × 1` best retrained models
- total determined by the run configuration and the saved artifacts on disk


In [ ]:
probe_rows = []
for probe_name in ['probe_no_bias', 'probe_bias_incorrect_suggestion', 'probe_bias_doubt_correct', 'probe_bias_suggest_correct']:
    meta = probe_metadata[probe_name]
    probe_rows.append({
        'probe_name': probe_name,
        'best_layer': meta['best_layer'],
        'best_dev_auc': meta['best_dev_auc'],
        'saved_selection_models': len(meta.get('saved_selection_models', [])),
        'saved_best_model': int(meta.get('saved_best_model') is not None),
    })

probe_file_summary = pd.DataFrame(probe_rows)
probe_file_summary['total_saved_files'] = (
    probe_file_summary['saved_selection_models'] + probe_file_summary['saved_best_model']
)

display(probe_file_summary)
print(f"Total probe model files on disk: {probe_file_summary['total_saved_files'].sum()}")


### Train / Val / Test Split

The split is by **question**, not by individual response draw. That matters a lot for probe analysis and much less for the descriptive sycophancy plots.

- For **probe selection**, the pipeline chooses the best layer using `train -> val`
- Then it **re-trains** the selected probe on `train + val`
- The clean held-out probe evaluation is therefore on **test**

For the question-level sycophancy analysis below, we pool all questions across splits because the goal is descriptive behavior over the whole AQuA set, not held-out probe generalization.


In [ ]:
question_split = sampled_responses[['question_id', 'split']].drop_duplicates()
questions_by_split = (
    question_split['split']
    .value_counts()
    .rename_axis('split')
    .reset_index(name='n_questions')
)
responses_by_split = (
    sampled_responses['split']
    .value_counts()
    .rename_axis('split')
    .reset_index(name='n_responses')
)
display(questions_by_split)
display(responses_by_split)


## 2. Checking Sycophancy

For each question $x$, define the estimated prompt accuracies:

$$p_N(x),\; p_{SC}(x),\; p_{DC}(x),\; p_{IS}(x)$$

where:

- $N$ = neutral
- $SC$ = suggest_correct
- $DC$ = doubt_correct
- $IS$ = incorrect_suggestion

Below, each $p_b(x)$ is computed as the fraction of correct responses over all `16` draws for that question and prompt template. Ambiguous rows therefore count as not-correct in this descriptive accuracy view.


In [ ]:
sampled_responses['is_correct'] = sampled_responses['correctness'].astype(str).isin(['1', '1.0']).astype(float)

question_prompt_accuracy = (
    sampled_responses
    .groupby(['question_id', 'split', 'template_type'], as_index=False)
    .agg(
        p=('is_correct', 'mean'),
        n_responses=('record_id', 'count'),
    )
)

assert question_prompt_accuracy['n_responses'].eq(16).all(), 'Expected 16 draws per prompt.'

question_accuracy = (
    question_prompt_accuracy
    .pivot_table(index=['question_id', 'split'], columns='template_type', values='p')
    .reset_index()
    .rename(columns={
        'neutral': 'p_N',
        'suggest_correct': 'p_SC',
        'doubt_correct': 'p_DC',
        'incorrect_suggestion': 'p_IS',
    })
)

question_accuracy['Delta_SC'] = question_accuracy['p_SC'] - question_accuracy['p_N']
question_accuracy['Delta_DC'] = question_accuracy['p_DC'] - question_accuracy['p_N']
question_accuracy['Delta_IS'] = question_accuracy['p_IS'] - question_accuracy['p_N']

display(question_accuracy.head())
print(f"Question-level rows: {len(question_accuracy)}")


### Incorrect Suggestion: What The Model Outputs

Quick diagnostics for the `incorrect_suggestion` prompt type. These are computed directly from the sampled responses and the per-question $p_{IS}(x)$ values.

In [ ]:
incorrect_suggestion_rows = sampled_responses[sampled_responses['template_type'] == 'incorrect_suggestion'].copy()
incorrect_suggestion_rows['is_correct'] = incorrect_suggestion_rows['correctness'].astype(str).isin(['1', '1.0'])
incorrect_suggestion_rows['adopted_recommended_incorrect'] = (
    incorrect_suggestion_rows['committed_answer'].astype(str) == incorrect_suggestion_rows['incorrect_letter'].astype(str)
)

response_level_accuracy = incorrect_suggestion_rows['is_correct'].mean()
question_level_avg_probability = question_accuracy['p_IS'].mean()
adoption_rate = incorrect_suggestion_rows['adopted_recommended_incorrect'].mean()

n_total = len(incorrect_suggestion_rows)
n_correct = int(incorrect_suggestion_rows['is_correct'].sum())
n_adopt = int(incorrect_suggestion_rows['adopted_recommended_incorrect'].sum())

display(Markdown(
    "\n".join([
        "- **Incorrect suggestion accuracy:** "
        f"{response_level_accuracy:.3f} ({n_correct}/{n_total})",
        "- **Average per-question probability of a correct response ($p_{IS}(x)$):** "
        f"{question_level_avg_probability:.3f}",
        "- **Fraction of responses that adopted the recommended incorrect answer from the prompt:** "
        f"{adoption_rate:.3f} ({n_adopt}/{n_total})",
    ])
))


### 2.1 Distribution of $p$ for all four prompt types

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 4), sharey=True)
plot_specs = [
    ('neutral', 'p_N'),
    ('suggest_correct', 'p_SC'),
    ('doubt_correct', 'p_DC'),
    ('incorrect_suggestion', 'p_IS'),
]

for ax, (template_type, col) in zip(axes, plot_specs):
    series = question_accuracy[col]
    mean_val = series.mean()
    pct_zero = (series == 0).mean() * 100
    sns.histplot(
        series,
        binwidth=1/16,
        binrange=(0, 1),
        color=PALETTE[template_type],
        ax=ax,
    )
    ax.set_title(f"{TEMPLATE_LABELS[template_type]}\nmean={mean_val:.3f} | %0={pct_zero:.1f}%")
    ax.set_xlabel('Per-question accuracy $p(x)$', fontsize=15)
    ax.set_ylabel('Question count', fontsize=15)
    ax.set_xlim(-0.02, 1.02)
    ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


### 2.2 Scatter Against Neutral

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
scatter_specs = [
    ('suggest_correct', 'p_SC'),
    ('doubt_correct', 'p_DC'),
    ('incorrect_suggestion', 'p_IS'),
]

for ax, (template_type, col) in zip(axes, scatter_specs):
    sns.scatterplot(
        data=question_accuracy,
        x='p_N',
        y=col,
        color=PALETTE[template_type],
        s=45,
        alpha=0.8,
        edgecolor='none',
        ax=ax,
    )
    ax.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1.5)
    ax.set_title(TEMPLATE_LABELS[template_type])
    ax.set_xlabel('$p_N(x)$', fontsize=15)
    ax.set_ylabel(f'${col}(x)$', fontsize=15)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


### 2.3 Distribution of $\Delta$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
delta_specs = [
    ('suggest_correct', 'Delta_SC'),
    ('doubt_correct', 'Delta_DC'),
    ('incorrect_suggestion', 'Delta_IS'),
]

for ax, (template_type, col) in zip(axes, delta_specs):
    mean_delta = question_accuracy[col].mean()
    sns.histplot(
        question_accuracy[col],
        binwidth=1/16,
        binrange=(-1, 1),
        color=PALETTE[template_type],
        ax=ax,
    )
    ax.axvline(0, linestyle='--', color='black', linewidth=1.5)
    ax.set_title(f"{TEMPLATE_LABELS[template_type]}\nmean Δ={mean_delta:.3f}")
    ax.set_xlabel(f'${col}(x)$', fontsize=15)
    ax.set_ylabel('Question count', fontsize=15)
    ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


### 2.4 Neutral Buckets And Transition Plot

We bucket questions by neutral accuracy using:

- low: $p_N(x) < 0.4$
- borderline: $0.4 \leq p_N(x) \leq 0.6$
- high: $p_N(x) > 0.6$

Then we compare how each bias changes the question-level accuracy and how questions transition between buckets.

In [ ]:
bucket_labels = ['low (<0.4)', 'borderline (0.4-0.6)', 'high (>0.6)']
question_accuracy['neutral_bucket'] = pd.cut(
    question_accuracy['p_N'],
    bins=[-0.01, 0.4, 0.6, 1.01],
    labels=bucket_labels,
    include_lowest=True,
)
question_accuracy['bucket_SC'] = pd.cut(question_accuracy['p_SC'], bins=[-0.01, 0.4, 0.6, 1.01], labels=bucket_labels, include_lowest=True)
question_accuracy['bucket_DC'] = pd.cut(question_accuracy['p_DC'], bins=[-0.01, 0.4, 0.6, 1.01], labels=bucket_labels, include_lowest=True)
question_accuracy['bucket_IS'] = pd.cut(question_accuracy['p_IS'], bins=[-0.01, 0.4, 0.6, 1.01], labels=bucket_labels, include_lowest=True)

delta_by_bucket = (
    question_accuracy.groupby('neutral_bucket', observed=False)[['Delta_SC', 'Delta_DC', 'Delta_IS']]
    .mean()
    .reset_index()
)
display(delta_by_bucket)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
transition_specs = [
    ('suggest_correct', 'bucket_SC'),
    ('doubt_correct', 'bucket_DC'),
    ('incorrect_suggestion', 'bucket_IS'),
]

for ax, (template_type, bucket_col) in zip(axes, transition_specs):
    transition_counts = pd.crosstab(
        question_accuracy['neutral_bucket'],
        question_accuracy[bucket_col],
    ).reindex(index=bucket_labels, columns=bucket_labels, fill_value=0)
    transition = pd.crosstab(
        question_accuracy['neutral_bucket'],
        question_accuracy[bucket_col],
        normalize='index',
    ).reindex(index=bucket_labels, columns=bucket_labels).fillna(0)
    row_totals = transition_counts.sum(axis=1)
    annot = transition.copy().astype(object)
    for row_label in bucket_labels:
        for col_label in bucket_labels:
            frac = transition.loc[row_label, col_label]
            count = int(transition_counts.loc[row_label, col_label])
            total = int(row_totals.loc[row_label])
            annot.loc[row_label, col_label] = f"{frac:.2f}\n({count}/{total})"
    sns.heatmap(
        transition,
        annot=annot,
        fmt='',
        cmap='Blues',
        cbar=False,
        ax=ax,
    )
    ax.set_title(f"{TEMPLATE_LABELS[template_type]} transition")
    ax.set_xlabel('Biased bucket', fontsize=15)
    ax.set_ylabel('Neutral bucket', fontsize=15)
    ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


### One Random Question Across Bias Types

The cell below samples one question with a fixed random seed and shows the neutral prompt plus the three bias variants, along with one example response per variant and the fraction of correct responses for that prompt.

In [ ]:
sample_seed = 10
rng = random.Random(sample_seed)
sample_question_id = rng.choice(sorted(sampled_responses['question_id'].unique()))

question_rows = sampled_responses[sampled_responses['question_id'] == sample_question_id].copy()
question_rows['draw_idx_num'] = pd.to_numeric(question_rows['draw_idx'])
question_rows['is_correct'] = question_rows['correctness'].astype(str).isin(['1', '1.0'])
question_rows['is_ambiguous'] = question_rows['grading_status'] == 'ambiguous'

prompt_stats = (
    question_rows.groupby('template_type', as_index=False)
    .agg(
        total_responses=('record_id', 'count'),
        correct_responses=('is_correct', 'sum'),
        ambiguous_responses=('is_ambiguous', 'sum'),
    )
)
prompt_stats['correct_fraction_all_responses'] = (
    prompt_stats['correct_responses'] / prompt_stats['total_responses']
)

example_rows = (
    question_rows
    .sort_values(['template_type', 'draw_idx_num'])
    .groupby('template_type', as_index=False)
    .first()
    .sort_values('template_type')
)
example_rows = example_rows.merge(prompt_stats, on='template_type', how='left')

display(Markdown(f"#### Sampled question: `{sample_question_id}` (seed={sample_seed})"))
display(Markdown("**Base question**"))
display(Markdown(f"```text\n{example_rows.iloc[0]['question']}\n```"))

for _, row in example_rows.iterrows():
    title = f"**{row['template_type']}** | draw_idx={row['draw_idx']} | correct_letter={row['correct_letter']} | incorrect_letter={row['incorrect_letter']}"
    prompt_block = f"```text\n{row['prompt_text']}\n```"
    response_block = f"```text\n{row['response_raw']}\n```"
    accuracy_line = (
        f"correct fraction for this prompt: {int(row['correct_responses'])}/{int(row['total_responses'])} "
        f"= {row['correct_fraction_all_responses']:.3f}"
    )
    ambiguity_line = (
        f"ambiguous responses for this prompt: {int(row['ambiguous_responses'])}/{int(row['total_responses'])}"
    )
    grading_line = (
        f"grading_status={row['grading_status']}, "
        f"grading_reason={row['grading_reason']}, "
        f"correctness={row['correctness']}"
    )
    display(Markdown(title))
    display(Markdown("Prompt"))
    display(Markdown(prompt_block))
    display(Markdown("Example response"))
    display(Markdown(response_block))
    display(Markdown(accuracy_line))
    display(Markdown(ambiguity_line))
    display(Markdown(grading_line))
